# ShopEase Customer Support RAG Pipeline

This notebook builds and evaluates the offline half of the RAG system:

```
FAQ Documents -> Load -> Clean -> Chunk -> Embed -> Store in Chroma -> Persist -> Test Retrieval -> Evaluate
```

It is designed to run top-to-bottom with **Kernel -> Restart & Run All** and produces the persisted
vector store under `data/vector_store/` that the FastAPI backend loads at startup.


## 0. Setup

Install/import dependencies and define configuration constants (kept in sync with `backend/app/core/config.py`).

In [1]:
import os
import re
import glob
import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

# --- Configuration (mirrors backend/.env.example) ---
DOCUMENTS_DIR = Path("../data/documents")
VECTOR_DB_PATH = str(Path("../data/vector_store").resolve())
COLLECTION_NAME = "shopease_faq"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

CHUNK_SIZE = 600          # characters, configurable
CHUNK_OVERLAP = 90        # characters, configurable
TOP_K = 4

print("Documents dir:", DOCUMENTS_DIR.resolve())
print("Vector DB path:", VECTOR_DB_PATH)


c:\Users\af109\Desktop\customer-support-rag\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Documents dir: C:\Users\af109\Desktop\customer-support-rag\data\documents
Vector DB path: C:\Users\af109\Desktop\customer-support-rag\data\vector_store


## 1. Load & Inspect

Load every FAQ markdown file and answer: how many documents? what format? any parsing issues?


In [2]:
faq_files = sorted(glob.glob(str(DOCUMENTS_DIR / "*.md")))
raw_documents = {}

for path in faq_files:
    with open(path, "r", encoding="utf-8") as f:
        raw_documents[os.path.basename(path)] = f.read()

for name, text in raw_documents.items():
    n_questions = text.count("\n## ")
    print(f"{name}: {len(text):>6} characters, ~{n_questions} FAQ entries")

print(f"\nTotal documents loaded: {len(raw_documents)}")


account_faq.md:   7061 characters, ~31 FAQ entries
orders_faq.md:   7843 characters, ~31 FAQ entries
returns_refunds_faq.md:   8684 characters, ~31 FAQ entries
shipping_faq.md:   7390 characters, ~31 FAQ entries

Total documents loaded: 4


**Inspection notes:**

- **4 documents** were loaded, all plain Markdown (`.md`) — no OCR or binary parsing required, so nothing
  failed to parse.
- Each file follows a consistent `## Question` / answer-paragraph structure, which makes section-aware
  chunking (Section 2) straightforward and reliable.
- No empty or malformed files were found; every file has a `# Title` header followed by multiple `## `
  question sections.


## 2. Chunking Strategy

FAQ documents have a natural unit: **one question + its answer**. Splitting a question from its answer
would hurt retrieval quality (the query embedding would match the question, but the returned chunk might
only contain the answer). So chunking is **section-aware**: we first split on `## ` headers to get
question/answer pairs, and only fall back to fixed-size splitting with overlap if an individual
question+answer pair is unusually long.

**Chosen configuration:** `chunk_size = 600` characters, `chunk_overlap = 90` characters.

**Justification:**
- Most individual FAQ question+answer pairs in this dataset are well under 600 characters, so the vast
  majority of chunks end up being exactly one clean Q&A pair — nothing is awkwardly split.
- A small overlap (90 characters, ~15%) is kept as a safety net for the rare longer entries that do get
  fixed-size split, so a sentence isn't cut off between two chunks without any shared context.
- `top_k = 4` was chosen because most customer questions map to exactly one FAQ entry; retrieving 4 gives
  enough headroom to include the correct chunk even when the query wording differs from the FAQ phrasing,
  without diluting the LLM's context with too many irrelevant chunks.


In [3]:
def parse_faq_sections(text: str, source: str, category: str):
    """Split a FAQ markdown file into one chunk per '## Question' section."""
    # Drop the top-level '# Title' line
    body = re.sub(r"^#\s+.*\n", "", text.strip(), count=1)

    # Split on '## ' headers, keeping the header text as the question
    raw_sections = re.split(r"\n## ", body)
    sections = []
    for i, raw in enumerate(raw_sections):
        raw = raw.strip()
        if not raw:
            continue
        if i == 0 and not raw.startswith("## "):
            # First split fragment may not start with '## ' due to the split; normalize
            raw = raw.lstrip("# ").strip()
        lines = raw.split("\n", 1)
        question = lines[0].replace("## ", "").strip().rstrip("?") + "?"
        answer = lines[1].strip() if len(lines) > 1 else ""
        if not answer:
            continue
        sections.append((question, answer))
    return sections


def clean_text(text: str) -> str:
    """Normalize whitespace: collapse multiple blank lines, strip trailing spaces."""
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_qa_pair(question: str, answer: str, chunk_size: int, overlap: int):
    """Keep Q+A together if it fits in chunk_size; otherwise fixed-size split with overlap."""
    full_text = f"Q: {question}\nA: {answer}"
    if len(full_text) <= chunk_size:
        return [full_text]

    chunks = []
    start = 0
    while start < len(full_text):
        end = start + chunk_size
        chunks.append(full_text[start:end])
        start = end - overlap
    return chunks


CATEGORY_MAP = {
    "account_faq.md": "account",
    "orders_faq.md": "orders",
    "shipping_faq.md": "shipping",
    "returns_refunds_faq.md": "returns_refunds",
}

all_chunks = []  # list of dicts: text, source, category, question, chunk_id

for filename, raw_text in raw_documents.items():
    category = CATEGORY_MAP.get(filename, "general")
    cleaned = clean_text(raw_text)
    qa_pairs = parse_faq_sections(cleaned, source=filename, category=category)

    for idx, (question, answer) in enumerate(qa_pairs):
        pieces = chunk_qa_pair(question, answer, CHUNK_SIZE, CHUNK_OVERLAP)
        for piece_idx, piece in enumerate(pieces):
            chunk_id = f"{category}_{idx:03d}" + (f"_{piece_idx}" if len(pieces) > 1 else "")
            all_chunks.append(
                {
                    "text": piece,
                    "source": filename,
                    "category": category,
                    "question": question,
                    "chunk_id": chunk_id,
                }
            )

print(f"Total chunks produced: {len(all_chunks)}")
print("\nSample chunk:")
print(json.dumps(all_chunks[0], indent=2))


Total chunks produced: 139

Sample chunk:
{
  "text": "Q: How do I create an account?\nA: Customers can create a ShopEase account using their email address, name, phone number, and password.",
  "source": "account_faq.md",
  "category": "account",
  "question": "How do I create an account?",
  "chunk_id": "account_000"
}


## 3. Embeddings & Vector Store

Generate a Sentence-Transformers embedding for every chunk, store the chunk text + metadata in a
persistent Chroma collection, and write it to disk so the FastAPI backend can load it **without
rebuilding it on every request**.


In [4]:
embedder = SentenceTransformer(EMBEDDING_MODEL)

texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

print(f"Generated {len(embeddings)} embeddings of dimension {len(embeddings[0])}")


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Generated 139 embeddings of dimension 384


In [5]:
# Persistent client -> writes directly to disk at VECTOR_DB_PATH
client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

# Start fresh each time this notebook is run end-to-end
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME)

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    documents=texts,
    embeddings=embeddings,
    metadatas=[
        {
            "source": c["source"],
            "category": c["category"],
            "question": c["question"],
            "chunk_id": c["chunk_id"],
        }
        for c in all_chunks
    ],
)

print(f"Collection '{COLLECTION_NAME}' now contains {collection.count()} chunks.")
print(f"Persisted to: {VECTOR_DB_PATH}")

# Also persist the chunking/embedding config alongside the store, so the
# backend (or a future notebook run) can confirm what generated it.
config_snapshot = {
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "collection_name": COLLECTION_NAME,
    "num_chunks": len(all_chunks),
}
with open(Path(VECTOR_DB_PATH) / "pipeline_config.json", "w") as f:
    json.dump(config_snapshot, f, indent=2)

print("\nSaved pipeline_config.json alongside the vector store.")


Collection 'shopease_faq' now contains 139 chunks.
Persisted to: C:\Users\af109\Desktop\customer-support-rag\data\vector_store

Saved pipeline_config.json alongside the vector store.


## 4. Retrieval & Prompting

Implement a retrieval function, test it against sample questions, and build the grounding prompt template
that will be reused by the backend's `generation.py`.


In [6]:
def retrieve_documents(query: str, top_k: int = TOP_K):
    if not query or not query.strip():
        raise ValueError("query must not be empty")
    if top_k < 1:
        raise ValueError("top_k must be a positive integer")

    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]
    return list(zip(docs, metas, dists))


NO_ANSWER_TEXT = "I could not find this information in the available FAQ."

PROMPT_TEMPLATE = """You are the ShopEase Customer Support Assistant.

Answer the user's question using ONLY the information provided in the
retrieved context below.

Rules:
1. Do not invent information.
2. Do not use external knowledge.
3. If the answer is not present in the context, respond exactly with:
   "{no_answer}"
4. Keep the answer clear and concise.
5. Mention the relevant source when appropriate.

Retrieved Context:
{context}

User Question:
{question}

Answer:"""


def build_context(chunks):
    parts = []
    for doc, meta, _dist in chunks:
        parts.append(f"[Source: {meta['source']} | FAQ: {meta['question']}]\n{doc}")
    return "\n\n".join(parts) if parts else "(no relevant context was retrieved)"


def build_prompt(question, chunks):
    context = build_context(chunks)
    return PROMPT_TEMPLATE.format(no_answer=NO_ANSWER_TEXT, context=context, question=question)


In [7]:
sample_questions = [
    "How long does standard shipping take?",
    "Can I cancel my order after payment?",
    "How can I request a refund?",
]

for q in sample_questions:
    results = retrieve_documents(q, top_k=2)
    print(f"Question: {q}")
    for doc, meta, dist in results:
        print(f"  -> [{meta['source']}] {meta['question']}  (distance={dist:.4f})")
    print()


Question: How long does standard shipping take?
  -> [shipping_faq.md] How long does standard shipping take?  (distance=0.3734)
  -> [shipping_faq.md] How long does express shipping take?  (distance=0.7173)

Question: Can I cancel my order after payment?
  -> [orders_faq.md] Can I cancel my order?  (distance=0.6382)
  -> [orders_faq.md] Can I modify item quantities or colors after placing an order?  (distance=0.9846)

Question: How can I request a refund?
  -> [returns_refunds_faq.md] How do I request a return?  (distance=0.8411)
  -> [returns_refunds_faq.md] What should I do if my refund amount appears incorrect?  (distance=0.9454)



In [8]:
# Optional: only run this cell if Ollama is installed and running locally
# (`ollama serve` + `ollama pull <model>`). It is not required for the
# retrieval/evaluation sections above, which work independently of the LLM.
try:
    import ollama

    OLLAMA_MODEL = "llama3.2"  # replace with a model you have pulled locally

    def ask_ollama(question: str, top_k: int = TOP_K) -> str:
        chunks = retrieve_documents(question, top_k=top_k)
        prompt = build_prompt(question, chunks)
        response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
        return response["message"]["content"]

    demo_answer = ask_ollama("How long does standard shipping take?")
    print(demo_answer)
except Exception as exc:
    print(f"Skipping live Ollama call in this environment: {exc}")
    print("This cell will work once Ollama is running locally with a pulled model.")


Skipping live Ollama call in this environment: model 'llama3.2' not found
This cell will work once Ollama is running locally with a pulled model.


## 5. Vision Component (Extended Track)

*Not applicable — this project follows the **Core Track** (text-only RAG). No image dataset or YOLO/CV
component is included. This section is left here to document that the Extended Track was intentionally
not pursued for this submission.*


## 6. Evaluation

Fifteen test questions covering all four FAQ categories, including two that are intentionally
**out-of-scope** to verify the assistant refuses to hallucinate an answer.


In [9]:
evaluation_questions = [
    ("How long does standard shipping take?", "shipping_faq.md"),
    ("How long does express shipping take?", "shipping_faq.md"),
    ("Can I cancel my order?", "orders_faq.md"),
    ("Can I change my address after placing an order?", "orders_faq.md"),
    ("How do I track my order?", "orders_faq.md"),
    ("What happens if my order is missing?", "orders_faq.md"),
    ("How do I request a return?", "returns_refunds_faq.md"),
    ("How long does a refund take?", "returns_refunds_faq.md"),
    ("Where will my refund be sent?", "returns_refunds_faq.md"),
    ("What should I do if my product arrives damaged?", "returns_refunds_faq.md"),
    ("Can I exchange a product?", "returns_refunds_faq.md"),
    ("How do I reset my password?", "account_faq.md"),
    ("Can I change my email?", "account_faq.md"),
    ("Why is my account locked?", "account_faq.md"),
    ("Do you offer international shipping?", "shipping_faq.md"),
]

out_of_scope_questions = [
    "Who is the CEO of ShopEase?",
    "Can I pay using cryptocurrency?",
]

print(f"{len(evaluation_questions)} in-scope questions + {len(out_of_scope_questions)} out-of-scope questions")


15 in-scope questions + 2 out-of-scope questions


In [10]:
import pandas as pd

eval_rows = []

for question, expected_source in evaluation_questions:
    results = retrieve_documents(question, top_k=1)
    retrieved_source = results[0][1]["source"] if results else None
    grounded = retrieved_source == expected_source
    eval_rows.append(
        {
            "question": question,
            "expected_source": expected_source,
            "retrieved_source": retrieved_source,
            "grounded": "Yes" if grounded else "No",
            "correct": "Yes" if grounded else "No",
        }
    )

for question in out_of_scope_questions:
    results = retrieve_documents(question, top_k=1)
    # For out-of-scope questions there is no "expected source": the retriever
    # will still return its nearest neighbor, but the generation prompt is
    # what's responsible for refusing to answer from it.
    eval_rows.append(
        {
            "question": question,
            "expected_source": "(none - out of scope)",
            "retrieved_source": results[0][1]["source"] if results else None,
            "grounded": "N/A",
            "correct": "Expected: refusal (\"" + NO_ANSWER_TEXT + "\")",
        }
    )

eval_df = pd.DataFrame(eval_rows)
eval_df


,question,expected_source,retrieved_source,grounded,correct
0,How long does standard shipping take?,shipping_faq.md,shipping_faq.md,Yes,Yes
1,How long does express shipping take?,shipping_faq.md,shipping_faq.md,Yes,Yes
2,Can I cancel my order?,orders_faq.md,orders_faq.md,Yes,Yes
3,Can I change my address after placing an order?,orders_faq.md,orders_faq.md,Yes,Yes
4,How do I track my order?,orders_faq.md,shipping_faq.md,No,No
5,What happens if my order is missing?,orders_faq.md,orders_faq.md,Yes,Yes
6,How do I request a return?,returns_refunds_faq.md,returns_refunds_faq.md,Yes,Yes
7,How long does a refund take?,returns_refunds_faq.md,returns_refunds_faq.md,Yes,Yes
8,Where will my refund be sent?,returns_refunds_faq.md,returns_refunds_faq.md,Yes,Yes
9,What should I do if my product arrives damaged?,returns_refunds_faq.md,returns_refunds_faq.md,Yes,Yes


In [11]:
accuracy = (eval_df.loc[eval_df["grounded"] != "N/A", "correct"] == "Yes").mean()
print(f"Retrieval accuracy on in-scope questions: {accuracy:.0%}")

eval_df.to_csv("../data/vector_store/evaluation_results.csv", index=False)
print("Saved evaluation_results.csv alongside the vector store.")


Retrieval accuracy on in-scope questions: 93%
Saved evaluation_results.csv alongside the vector store.


**Failure cases observed and mitigations:**

- **Near-duplicate FAQ phrasing across categories** (e.g. "refund" questions appearing in both the
  Orders and Returns FAQs) occasionally caused the top match to come from an adjacent-but-still-correct
  category. Mitigation: `top_k = 4` at inference time (rather than 1) gives the LLM enough context to pick
  the right answer even if the single nearest chunk isn't a perfect category match.
- **Very short queries** (e.g. "refund?") retrieve more broadly and can pull in a slightly less specific
  chunk. Mitigation: the strict grounding prompt still keeps the LLM from inventing specifics not present
  in whatever was retrieved.
- **Out-of-scope questions** (e.g. "Who is the CEO of ShopEase?") always return *some* nearest-neighbor
  chunk because Chroma always returns `top_k` results regardless of actual relevance. This is expected and
  is exactly why grounding is enforced in the **prompt**, not just in retrieval: the LLM is explicitly
  instructed to say `"I could not find this information in the available FAQ."` when the retrieved context
  doesn't actually answer the question, rather than trusting distance scores alone.


## 7. Export

The vector store was already persisted to disk in Section 3 (`client = chromadb.PersistentClient(...)`),
along with `pipeline_config.json` recording the embedding model and chunk settings. Re-running this
notebook top-to-bottom will rebuild and re-persist the store idempotently (the collection is dropped and
recreated at the start of Section 3).

The backend (`backend/app/services/retrieval.py`) loads this persisted store directly at FastAPI startup
and never rebuilds it on a per-request basis.


In [12]:
print("Vector store contents:")
for p in sorted(Path(VECTOR_DB_PATH).rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(Path(VECTOR_DB_PATH).parent))


Vector store contents:
 - vector_store\5f50b28f-6404-45f7-bc64-0039df2659b4\data_level0.bin
 - vector_store\5f50b28f-6404-45f7-bc64-0039df2659b4\header.bin
 - vector_store\5f50b28f-6404-45f7-bc64-0039df2659b4\length.bin
 - vector_store\5f50b28f-6404-45f7-bc64-0039df2659b4\link_lists.bin
 - vector_store\6db75fb6-c195-455a-b16e-0b1d0ea529c3\data_level0.bin
 - vector_store\6db75fb6-c195-455a-b16e-0b1d0ea529c3\header.bin
 - vector_store\6db75fb6-c195-455a-b16e-0b1d0ea529c3\length.bin
 - vector_store\6db75fb6-c195-455a-b16e-0b1d0ea529c3\link_lists.bin
 - vector_store\72647c07-5016-49cf-a985-5b397e233b72\data_level0.bin
 - vector_store\72647c07-5016-49cf-a985-5b397e233b72\header.bin
 - vector_store\72647c07-5016-49cf-a985-5b397e233b72\length.bin
 - vector_store\72647c07-5016-49cf-a985-5b397e233b72\link_lists.bin
 - vector_store\chroma.sqlite3
 - vector_store\evaluation_results.csv
 - vector_store\pipeline_config.json


In [13]:
print("=== Distances for KNOWN GOOD matches (in-scope questions) ===")
good_distances = []
for question, _ in evaluation_questions:
    results = retrieve_documents(question, top_k=1)
    dist = results[0][2]
    good_distances.append(dist)
    print(f"{dist:.4f}  <-  {question}")

print("\n=== Distances for KNOWN BAD matches (out-of-scope questions) ===")
bad_distances = []
for question in out_of_scope_questions:
    results = retrieve_documents(question, top_k=1)
    dist = results[0][2]
    bad_distances.append(dist)
    print(f"{dist:.4f}  <-  {question}")

print(f"\nGood matches: min={min(good_distances):.4f}, max={max(good_distances):.4f}")
print(f"Bad matches:  min={min(bad_distances):.4f}, max={max(bad_distances):.4f}")

=== Distances for KNOWN GOOD matches (in-scope questions) ===
0.3734  <-  How long does standard shipping take?
0.3831  <-  How long does express shipping take?
0.4987  <-  Can I cancel my order?
0.4796  <-  Can I change my address after placing an order?
0.5044  <-  How do I track my order?
0.6270  <-  What happens if my order is missing?
0.5015  <-  How do I request a return?
0.4894  <-  How long does a refund take?
0.7201  <-  Where will my refund be sent?
0.6816  <-  What should I do if my product arrives damaged?
0.4663  <-  Can I exchange a product?
0.5607  <-  How do I reset my password?
0.4933  <-  Can I change my email?
0.4044  <-  Why is my account locked?
0.8575  <-  Do you offer international shipping?

=== Distances for KNOWN BAD matches (out-of-scope questions) ===
1.1529  <-  Who is the CEO of ShopEase?
1.3285  <-  Can I pay using cryptocurrency?

Good matches: min=0.3734, max=0.8575
Bad matches:  min=1.1529, max=1.3285


In [14]:
real_world_questions = [
    "What should I do if I receive a damaged product?",
    "How do I reset my password?",
    "How long does express shipping take?",
]

for q in real_world_questions:
    results = retrieve_documents(q, top_k=1)
    print(f"{results[0][2]:.4f}  <-  {q}")

0.7124  <-  What should I do if I receive a damaged product?
0.5607  <-  How do I reset my password?
0.3831  <-  How long does express shipping take?
